# 07 — Advanced catalogue search

`Catalogue.search()` filters the catalogue using required instrument properties and spectral-data availability. It returns a frozen `Instruments` collection, so search results support the same mapping and attribute access as `xeo.instruments`.

In [ ]:
import xeo

## Search one property

Most search values use exact, case-sensitive matching. The searchable catalogue properties are `id`, `name`, `acronym`, `type`, `platform_type`, `platform`, `operator`, `start_date`, `status`, `availability`, and `references`. The `start_date` property also supports date intervals, as shown below.

In [ ]:
satellite_instruments = xeo.catalogue.search(platform_type="satellite")

print(type(satellite_instruments))
print(f"Matches: {len(satellite_instruments)}")
print(list(satellite_instruments))

## Use a list for OR matching

A list means “match any of these values.” For list-valued instrument properties such as `operator`, an instrument matches when any requested value is present. This search returns instruments operated by either ESA or NASA.

In [ ]:
agency_instruments = xeo.catalogue.search(operator=["ESA", "NASA"])

[
    (instrument.id, instrument.operator)
    for instrument in agency_instruments.values()
]

Lists work for other properties too, whether the underlying property is scalar or list-valued.

In [ ]:
selected_platforms = xeo.catalogue.search(
    platform=["Sentinel-2A", "Landsat 8"]
)
[(instrument.id, instrument.platform) for instrument in selected_platforms.values()]

## Search an inclusive start-date interval

Exact launch dates can be restrictive, so `start_date` accepts an inclusive interval formatted as `YYYY-MM-DD/YYYY-MM-DD`. An instrument matches when its start date falls on or between the two boundaries.

In [ ]:
started_between_2000_and_2003 = xeo.catalogue.search(
    start_date="2000-01-01/2003-01-01"
)
[
    (instrument.id, instrument.start_date)
    for instrument in started_between_2000_and_2003.values()
]

Date intervals retain the usual list-based OR behavior and can be mixed with exact dates.

In [ ]:
selected_start_dates = xeo.catalogue.search(
    start_date=["1999-04-15/1999-12-18", "2002-05-04"]
)
[(instrument.id, instrument.start_date) for instrument in selected_start_dates.values()]

## Combine properties with AND

Different keyword arguments are combined with AND. The following search requires every returned instrument to satisfy all four conditions.

In [ ]:
operational_multispectral = xeo.catalogue.search(
    operator=["ESA", "NASA"],
    platform_type="satellite",
    type="multispectral",
    status="operational",
)
list(operational_multispectral)

## Filter by spectral-data availability

In addition to required catalogue properties, `has_bands` and `has_srf` accept boolean values.

In [ ]:
instruments_with_srf = xeo.catalogue.search(has_srf=True)
instruments_without_srf = xeo.catalogue.search(has_srf=False)

print("With SRF:", len(instruments_with_srf))
print("Without SRF:", len(instruments_without_srf))
print(list(instruments_with_srf))

Boolean availability filters can be combined with metadata filters.

In [ ]:
public_satellites_with_srf = xeo.catalogue.search(
    platform_type="satellite",
    availability="public",
    has_bands=True,
    has_srf=True,
)
list(public_satellites_with_srf)

## Work with the result collection

Search results contain the same `Instrument` objects as the full catalogue and support both lookup styles.

In [ ]:
result = xeo.catalogue.search(id=["MSI_S2A", "OLI_L8"])

print(result.MSI_S2A)
print(result["OLI_L8"])
print(result.MSI_S2A is xeo.instruments.MSI_S2A)